# 03. Aggregates

**Цель:** на основе подготовленных данных построитькомпактные
агрегированные таблицы, которые дальее будут записаны в PostgreSQL.

**Витрины:**
1. `funnel_daily` — воронка типов событий по дням
2. `category_sales` — выручка и продажи по категориям и брендам
3. `user_metrics` — метрики активности и трат на пользователя
4. `cart_abandonment` — товары в корзине без покупки
5. `time_patterns` — активность по часам и дням недели
6. `daily_summary` — ключевые метрики по дням

## 0. Импорты и инициализациия

In [1]:
import sys
sys.path.append("..")

from utils.spark_session import get_spark_session
from utils.config import PROCESSED_DATA_DIR

spark = get_spark_session()

## 1. Чтение данных

In [5]:
events_path = f"{PROCESSED_DATA_DIR}/events_clean.parquet"

df = spark.read.parquet(events_path)
df.createOrReplaceTempView("events")

AGGREGATES_DIR = f"{PROCESSED_DATA_DIR}/aggregates"

## 2. Воронка по дням

Количество событий каждого типа.

In [6]:
funnel_daily = spark.sql("""
    SELECT
        event_date,
        event_type,
        COUNT(*) AS event_count
    FROM events
    GROUP BY event_date, event_type
    ORDER BY event_date, event_type
""")

funnel_daily.show(10, truncate=False)

+----------+----------+-----------+
|event_date|event_type|event_count|
+----------+----------+-----------+
|2019-10-01|cart      |16193      |
|2019-10-01|purchase  |19305      |
|2019-10-01|view      |1208217    |
|2019-10-02|cart      |16871      |
|2019-10-02|purchase  |19469      |
|2019-10-02|view      |1154513    |
|2019-10-03|cart      |18702      |
|2019-10-03|purchase  |19255      |
|2019-10-03|view      |1088667    |
|2019-10-04|cart      |42657      |
+----------+----------+-----------+
only showing top 10 rows



In [7]:
funnel_daily.write.mode("overwrite").parquet(f"{AGGREGATES_DIR}/funnel_daily.parquet")

## 3. Выручка и продажи по категориям и брендам

Считаем только покупки, исключая строки с флагом `is_price_missing`.
Используем как среднее, так и медиану, для наглядности,
так как распределение является скошенным.

In [8]:
category_sales = spark.sql("""
    SELECT
        category_l1,
        category_l2,
        brand,
        COUNT(*) AS purchase_count,
        ROUND(SUM(price), 2) AS total_revenue,
        ROUND(AVG(price), 2) AS avg_price,
        ROUND(PERCENTILE_APPROX(price, 0.5), 2) AS median_price
    FROM events
    WHERE event_type = 'purchase'
      AND is_price_missing = false
    GROUP BY category_l1, category_l2, brand
    ORDER BY total_revenue DESC
""")

category_sales.show(10, truncate=False)

+-----------+-----------+-------+--------------+--------------+---------+------------+
|category_l1|category_l2|brand  |purchase_count|total_revenue |avg_price|median_price|
+-----------+-----------+-------+--------------+--------------+---------+------------+
|electronics|smartphone |apple  |115334        |1.0254061706E8|889.08   |808.7       |
|electronics|smartphone |samsung|143108        |3.736897717E7 |261.12   |223.96      |
|electronics|smartphone |xiaomi |38769         |8058377.17    |207.86   |190.22      |
|NULL       |NULL       |NULL   |38294         |5375229.31    |140.37   |87.0        |
|electronics|smartphone |huawei |21880         |4702494.51    |214.92   |169.48      |
|electronics|video      |samsung|8968          |4420238.54    |492.89   |424.62      |
|computers  |notebook   |acer   |6016          |3254703.11    |541.01   |500.89      |
|NULL       |NULL       |lucente|11576         |3123438.96    |269.82   |215.71      |
|electronics|clocks     |apple  |7291      

In [9]:
category_sales.write.mode("overwrite").parquet(f"{AGGREGATES_DIR}/category_sales.parquet")

## 4. Активность и траты пользователей

Базовый RFM метрики

In [10]:
user_metrics = spark.sql("""
    SELECT
        user_id,
        COUNT(DISTINCT user_session) AS session_count,
        SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchase_count,
        ROUND(SUM(CASE WHEN event_type = 'purchase' THEN price ELSE 0 END), 2) AS total_spent,
        ROUND(AVG(CASE WHEN event_type = 'purchase' AND is_price_missing = false THEN price END), 2) AS avg_spent,
        ROUND(PERCENTILE_APPROX(CASE WHEN event_type = 'purchase' AND is_price_missing = false THEN price END, 0.5), 2) AS median_spent,
        MAX(event_date) AS last_active_date
    FROM events
    GROUP BY user_id
""")

user_metrics.show(10, truncate=False)

+---------+-------------+--------------+-----------+---------+------------+----------------+
|user_id  |session_count|purchase_count|total_spent|avg_spent|median_spent|last_active_date|
+---------+-------------+--------------+-----------+---------+------------+----------------+
|64078358 |1            |0             |0.0        |NULL     |NULL        |2019-10-13      |
|183503497|1            |0             |0.0        |NULL     |NULL        |2019-10-02      |
|210507313|1            |0             |0.0        |NULL     |NULL        |2019-10-18      |
|222907508|1            |0             |0.0        |NULL     |NULL        |2019-10-07      |
|237271696|1            |0             |0.0        |NULL     |NULL        |2019-10-07      |
|240522111|18           |0             |0.0        |NULL     |NULL        |2019-10-23      |
|249183593|1            |0             |0.0        |NULL     |NULL        |2019-10-12      |
|255544096|5            |0             |0.0        |NULL     |NULL    

In [11]:
user_metrics.write.mode("overwrite").parquet(f"{AGGREGATES_DIR}/user_metrics.parquet")

## 5. Товары в корзине без покупок

Считаем по `user_session` + `product_id` — товар мог быть добавлен и куплен в рамках одной сессии.

In [12]:
cart_abandonment = spark.sql("""
    WITH cart_events AS (
        SELECT DISTINCT user_session, product_id, category_l1, brand
        FROM events
        WHERE event_type = 'cart'
    ),
    purchase_events AS (
        SELECT DISTINCT user_session, product_id
        FROM events
        WHERE event_type = 'purchase'
    )
    SELECT
        c.category_l1,
        c.brand,
        COUNT(*) AS abandoned_count
    FROM cart_events c
    LEFT ANTI JOIN purchase_events p
        ON c.user_session = p.user_session AND c.product_id = p.product_id
    GROUP BY c.category_l1, c.brand
    ORDER BY abandoned_count DESC
""")

cart_abandonment.show(10, truncate=False)

+-----------+-------+---------------+
|category_l1|brand  |abandoned_count|
+-----------+-------+---------------+
|electronics|samsung|81623          |
|electronics|apple  |70314          |
|electronics|xiaomi |29687          |
|electronics|huawei |13076          |
|NULL       |xiaomi |8711           |
|appliances |samsung|7087           |
|electronics|oppo   |5954           |
|computers  |acer   |3627           |
|NULL       |NULL   |3409           |
|appliances |lg     |3127           |
+-----------+-------+---------------+
only showing top 10 rows



In [13]:
cart_abandonment.write.mode("overwrite").parquet(f"{AGGREGATES_DIR}/cart_abandonment.parquet")

## 6. Активность по часам и дням недели

Количество событий в каждый час каждого дня недели.

In [14]:
time_patterns = spark.sql("""
    SELECT
        day_of_week,
        event_hour,
        event_type,
        COUNT(*) AS event_count
    FROM events
    GROUP BY day_of_week, event_hour, event_type
    ORDER BY day_of_week, event_hour
""")

time_patterns.show(10, truncate=False)

+-----------+----------+----------+-----------+
|day_of_week|event_hour|event_type|event_count|
+-----------+----------+----------+-----------+
|1          |0         |cart      |463        |
|1          |0         |purchase  |378        |
|1          |0         |view      |39986      |
|1          |1         |cart      |1012       |
|1          |1         |view      |79546      |
|1          |1         |purchase  |756        |
|1          |2         |view      |133881     |
|1          |2         |cart      |2228       |
|1          |2         |purchase  |1813       |
|1          |3         |view      |186988     |
+-----------+----------+----------+-----------+
only showing top 10 rows



In [15]:
time_patterns.write.mode("overwrite").parquet(f"{AGGREGATES_DIR}/time_patterns.parquet")

## 7. Ключевые метрики по дням

Одна строка на день (31 строка на весь октябрь) с основными KPI — удобно для трендового графика.

In [16]:
daily_summary = spark.sql("""
    SELECT
        event_date,
        COUNT(*) AS total_events,
        COUNT(DISTINCT user_id) AS unique_users,
        COUNT(DISTINCT user_session) AS unique_sessions,
        SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchase_count,
        ROUND(SUM(CASE WHEN event_type = 'purchase' THEN price ELSE 0 END), 2) AS revenue,
        ROUND(
            SUM(CASE WHEN event_type = 'purchase' THEN price ELSE 0 END)
            / NULLIF(SUM(CASE WHEN event_type = 'purchase' AND is_price_missing = false THEN 1 ELSE 0 END), 0),
        2) AS avg_check
    FROM events
    GROUP BY event_date
    ORDER BY event_date
""")

daily_summary.show(10, truncate=False)

+----------+------------+------------+---------------+--------------+----------+---------+
|event_date|total_events|unique_users|unique_sessions|purchase_count|revenue   |avg_check|
+----------+------------+------------+---------------+--------------+----------+---------+
|2019-10-01|1243715     |190188      |268737         |19305         |6275579.06|325.08   |
|2019-10-02|1190853     |184965      |261252         |19469         |6213628.53|319.15   |
|2019-10-03|1126624     |170668      |241086         |19255         |6233782.98|323.75   |
|2019-10-04|1415939     |209410      |300651         |27039         |8623058.19|318.91   |
|2019-10-05|1329334     |194958      |276914         |23492         |7341094.46|312.49   |
|2019-10-06|1317164     |193210      |273034         |22169         |6737258.17|303.9    |
|2019-10-07|1199974     |186951      |263437         |21378         |6348189.06|296.95   |
|2019-10-08|1370077     |221204      |308348         |23071         |6819701.26|295.6    |

In [17]:
daily_summary.write.mode("overwrite").parquet(f"{AGGREGATES_DIR}/daily_summary.parquet")

## Итог

Все шесть витрин сохранены. На следующем шаге прочитаем
каждую витрину из Parquet и записышем в PostgreSQL.